# Aula 03 — Processamento de Linguagem Natural

Células na ordem dos slides; algumas dependem de variáveis de células anteriores.

## Slide 15 — BOW MANUALMENTE EM PYTHON SEM LIBS, PARA ENTENDER POR DENTRO

*Bloco 01*

In [1]:
# BoW manual — 30 linhas para entender o que sklearn faz
from collections import Counter
corpus = [
    'Adorei o produto, entrega rápida',
    'Produto chegou quebrado',
    'Entrega rápida, atendimento bom',
]
# 1. Tokenizar (simplificado)
docs_tokens = [doc.lower().replace(',','').split() for doc in corpus]
# 2. Construir vocabulário (palavras únicas ordenadas)
vocab = sorted(set(t for doc in docs_tokens for t in doc))
print(f'Vocabulário ({len(vocab)} palavras):', vocab)
# 3. Construir matriz documento-termo
matriz = []
for tokens in docs_tokens:
    contagem = Counter(tokens)
    linha = [contagem.get(p, 0) for p in vocab]
    matriz.append(linha)
import pandas as pd
df = pd.DataFrame(matriz, columns=vocab)
print(df)

Vocabulário (9 palavras): ['adorei', 'atendimento', 'bom', 'chegou', 'entrega', 'o', 'produto', 'quebrado', 'rápida']
   adorei  atendimento  bom  chegou  entrega  o  produto  quebrado  rápida
0       1            0    0       0        1  1        1         0       1
1       0            0    0       1        0  0        1         1       0
2       0            1    1       0        1  0        0         0       1


## Slide 16 — BOW COM SCIKIT-LEARN COUNTVECTORIZER FAZ TUDO

*Bloco 02*

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
corpus = [
    'Adorei o produto, entrega rápida e qualidade boa.',
    'Produto chegou quebrado, qualidade péssima.',
    'Entrega rápida, atendimento bom, recomendo!',
    'Produto bom mas atendimento péssimo.',
]
# CountVectorizer: tokenização + vocabulário + matriz
vectorizer = CountVectorizer(lowercase=True, max_features=20)
X = vectorizer.fit_transform(corpus)
vocab = vectorizer.get_feature_names_out()
df = pd.DataFrame(X.toarray(), columns=vocab)
print(df)
print(f'Tamanho do vocabulário: {len(vocab)}')
print(f'Esparsidade: {1 - X.nnz / (X.shape[0] * X.shape[1]):.2%}')

   adorei  atendimento  boa  bom  chegou  entrega  mas  produto  péssima  \
0       1            0    1    0       0        1    0        1        0   
1       0            0    0    0       1        0    0        1        1   
2       0            1    0    1       0        1    0        0        0   
3       0            1    0    1       0        0    1        1        0   

   péssimo  qualidade  quebrado  recomendo  rápida  
0        0          1         0          0       1  
1        0          1         1          0       0  
2        0          0         0          1       1  
3        1          0         0          0       0  
Tamanho do vocabulário: 14
Esparsidade: 62.50%


## Slide 17 — SAÍDA TÍPICA DO COUNTVECTORIZER MATRIZ DOCUMENTO-TERMO EM AÇÃO

*Bloco 03*

In [4]:
# Saída ilustrativa:
#    adorei  atendimento  bom  boa  chegou  entrega  mas  o  ...
# 0       1            0    0    1       0        1    0  1  ...
# 1       0            0    0    0       1        0    0  0  ...
# 2       0            1    1    0       0        1    0  0  ...
# 3       0            1    1    0       0        0    1  0  ...
# Tamanho do vocabulário: 17
# Esparsidade: 73.53%
# Cada linha: um documento
# Cada coluna: uma palavra do vocabulário
# Valores: contagem da palavra no documento
# Cuidado: lowercase=True é o default
# Outros parâmetros úteis:
#   - stop_words: lista para remover
#   - max_features: limita N palavras mais frequentes
#   - min_df / max_df: filtros por frequência

## Slide 18 — ESPARSIDADE MATRIZ CHEIA DE ZEROS

*Bloco 04*

In [5]:
# Visualizando a esparsidade
print(f'Shape: {X.shape}')                # (10000, 30000)
print(f'Total de células: {X.shape[0] * X.shape[1]:,}')  # 300,000,000
print(f'Não-zeros: {X.nnz:,}')             # ~1,500,000
print(f'Esparsidade: {1 - X.nnz/(X.shape[0]*X.shape[1]):.2%}')
# Esparsidade: 99.50%
# Armazenar matriz cheia em numpy: ~2.4 GB (300M × 8 bytes)
# Armazenar como esparsa: ~36 MB (1.5M × 24 bytes por trio)

Shape: (4, 14)
Total de células: 56
Não-zeros: 21
Esparsidade: 62.50%


## Slide 19 — SCIPY.SPARSE

*Bloco 05*

In [6]:
# [notebook] rótulos do corpus de 4 reviews do slide 16 (1=pos, 0=neg)
y = [1, 0, 1, 0]
from scipy.sparse import csr_matrix
import numpy as np
# X já vem como matriz esparsa do CountVectorizer
print(type(X))  # scipy.sparse.csr_matrix
# Usar com modelos de sklearn — todos suportam esparsa nativamente
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X, y)  # ✅ Funciona direto, sem converter
# Converter para denso APENAS se precisar visualizar
# (cuidado: pode estourar memória em corpora grandes!)
X_dense = X.toarray()  # ⚠️ Use só com X pequeno

<class 'scipy.sparse._csr.csr_matrix'>


## Slide 31 — TRIGRAMS (N=3)

*Bloco 06*

In [13]:
# Trigrams capturam expressões idiomáticas e contexto mais longo
from sklearn.feature_extraction.text import CountVectorizer
textos = [
    'não gostei nada do produto',
    'muito bom mesmo, recomendo',
    'foi de jeito nenhum',
]
# ngram_range=(3, 3) → APENAS trigrams
vec = CountVectorizer(ngram_range=(3, 3))
X = vec.fit_transform(textos)
print(vec.get_feature_names_out())
# Saída:
# ['de jeito nenhum', 'do produto'... 'mesmo recomendo',
#  'muito bom mesmo', 'não gostei nada']
# Trigrams capturam:
# - Expressões fixas: 'de jeito nenhum'
# - Modificadores intensivos: 'muito bom mesmo'
# - Negações compostas: 'não gostei nada'

['bom mesmo recomendo' 'de jeito nenhum' 'foi de jeito' 'gostei nada do'
 'muito bom mesmo' 'nada do produto' 'não gostei nada']


## Slide 32 — RANGE (1, 2): UNIGRAMS + BIGRAMS

*Bloco 07*

In [14]:
# CountVectorizer com unigrams + bigrams
vec = CountVectorizer(ngram_range=(1, 2), min_df=2)
X = vec.fit_transform(corpus)
# Vocabulário agora inclui AMBOS:
#   ['adorei', 'adorei produto', 'bom', 'entrega',
#    'entrega rápida', 'gostei', 'não', 'não gostei',
#    'produto', 'rápida', ...]
# Em sentiment analysis em PT-BR:
# - Unigrams capturam 'gostei', 'péssimo', 'maravilhoso'
# - Bigrams capturam 'não gostei', 'muito bom', 'super ruim'
print(f'Total de features: {len(vec.get_feature_names_out())}')

Total de features: 3


## Slide 33 — IMPLEMENTAÇÃO PRÁTICA

*Bloco 08*

In [15]:
from sklearn.feature_extraction.text import CountVectorizer
corpus = [
    'Adorei o produto, super recomendo',
    'Não gostei muito do produto',
    'Produto bom mesmo, vale o preço',
    'Muito ruim, não comprem',
]
# Comparando ngram_range diferentes
for n_range in [(1, 1), (1, 2), (1, 3), (2, 2)]:
    vec = CountVectorizer(ngram_range=n_range)
    X = vec.fit_transform(corpus)
    print(f'ngram_range={n_range}: {X.shape[1]} features')
# Saída típica:
# ngram_range=(1, 1): 18 features    ← unigrams só
# ngram_range=(1, 2): 35 features    ← +17 bigrams
# ngram_range=(1, 3): 50 features    ← +15 trigrams
# ngram_range=(2, 2): 17 features    ← APENAS bigrams

ngram_range=(1, 1): 14 features
ngram_range=(1, 2): 28 features
ngram_range=(1, 3): 38 features
ngram_range=(2, 2): 14 features


## Dados de exemplo (adicionado no notebook)

Os slides seguintes usam `reviews_corpus` e `labels`, que não aparecem nos slides.

In [10]:
# [notebook] Dados sintéticos de exemplo — os slides assumem um corpus "já preparado" (reviews + labels)
import random
random.seed(42)
_pos = ["Adorei o produto", "Excelente qualidade", "Entrega rápida e bem embalado", "Recomendo muito",
        "Funciona perfeitamente", "Ótimo atendimento", "Superou minhas expectativas", "Produto maravilhoso"]
_neg = ["Péssimo produto", "Chegou quebrado", "Entrega atrasada", "Não recomendo",
        "Veio com defeito", "Atendimento horrível", "Qualidade ruim", "Muito decepcionado"]
_neu = ["comprei na semana passada", "o preço estava bom", "a embalagem era simples", "usei por alguns dias",
        "chegou em casa ontem", "a cor é como na foto"]
reviews_corpus, labels = [], []
for _ in range(400):
    y_ = random.randint(0, 1)
    frases = random.sample(_pos if y_ else _neg, 2) + random.sample(_neu, 1)
    random.shuffle(frases)
    reviews_corpus.append(". ".join(frases) + ".")
    labels.append(y_)
reviews = reviews_corpus  # nome usado na aula 05
print(len(reviews_corpus), "reviews | positivas:", sum(labels))

400 reviews | positivas: 208


## Slide 35 — FILTROS: MIN_DF E MAX_DF

*Bloco 09*

In [16]:
# Filtros para podar n-grams ruidosos ou inúteis
vec = CountVectorizer(
    ngram_range=(1, 2),
    min_df=5,        # ignora termos em MENOS de 5 documentos
    max_df=0.85,     # ignora termos em MAIS de 85% dos documentos
    max_features=10000,  # limita vocabulário aos 10k mais frequentes
)
X = vec.fit_transform(reviews_corpus)  # [notebook] corpus maior, p/ min_df=5 fazer sentido
# Por que filtrar?
# - min_df remove n-grams raros (typos, palavras únicas) → reduz ruído
# - max_df remove palavras universais (alternativa a stop words)
# - max_features cap absoluto no tamanho do vocabulário
# Exemplo: corpus de 10k docs
# Sem filtros: ~600k features (lento!)
# Com filtros acima: ~10k features (manejável)

## Slide 38 — BIGRAMS + SENTIMENT ANALYSIS EM PT-BR

*Bloco 10*

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
# Corpus de reviews + labels (0=neg, 1=pos)
# reviews_corpus = [...]; labels = [...]
# Sem bigrams (unigrams puros)
bow_uni = CountVectorizer(ngram_range=(1, 1), min_df=5)
X_uni = bow_uni.fit_transform(reviews_corpus)
f1_uni = cross_val_score(LogisticRegression(max_iter=1000),
                          X_uni, labels, cv=5, scoring='f1').mean()
# Com unigrams + bigrams
bow_bi = CountVectorizer(ngram_range=(1, 2), min_df=5)
X_bi = bow_bi.fit_transform(reviews_corpus)
f1_bi = cross_val_score(LogisticRegression(max_iter=1000),
                         X_bi, labels, cv=5, scoring='f1').mean()
print(f'Unigrams puros:    F1 = {f1_uni:.3f}')
print(f'Unigrams + bigrams: F1 = {f1_bi:.3f}')
print(f'Ganho: +{(f1_bi - f1_uni)*100:.1f} pontos')
# Tipicamente: ganho de 2-5 pontos em sentiment PT-BR

Unigrams puros:    F1 = 1.000
Unigrams + bigrams: F1 = 1.000
Ganho: +0.0 pontos


## Slide 48 — TF: TERM FREQUENCY

*Bloco 11*

In [18]:
def tf(palavra, doc):
    """Term Frequency normalizado pelo tamanho do doc."""
    palavras = doc.split()
    return palavras.count(palavra) / len(palavras)
doc = 'o produto bom o produto rápido'
# 6 palavras totais
print(tf('o', doc))         # 2/6 = 0.333
print(tf('produto', doc))   # 2/6 = 0.333
print(tf('bom', doc))       # 1/6 = 0.167
print(tf('rápido', doc))    # 1/6 = 0.167

0.3333333333333333
0.3333333333333333
0.16666666666666666
0.16666666666666666


## Slide 49 — IDF: INVERSE DOCUMENT FREQUENCY

*Bloco 12*

In [17]:
import numpy as np
def idf(palavra, corpus):
    """Inverse Document Frequency."""
    df = sum(1 for doc in corpus if palavra in doc.split())
    return np.log(len(corpus) / df) if df > 0 else 0
corpus = [
    'o produto bom',
    'o produto ruim',
    'o produto quebrado',
    'o produto rápido',
]
print(idf('o', corpus))         # log(4/4) = 0     ← em TODO doc
print(idf('produto', corpus))   # log(4/4) = 0     ← em TODO doc
print(idf('bom', corpus))       # log(4/1) = 1.39  ← só em 1 doc

0.0
0.0
1.3862943611198906


## Slide 51 — TF-IDF MANUALMENTE

*Bloco 13*

In [19]:
import numpy as np
import pandas as pd
corpus = [
    'produto bom rapida entrega',
    'produto ruim quebrado',
    'entrega rapida atendimento bom',
    'atendimento ruim produto bom',
]
vocab = sorted(set(p for doc in corpus for p in doc.split()))
def tf(p, doc):
    palavras = doc.split()
    return palavras.count(p) / len(palavras)
def idf(p, corpus):
    df = sum(1 for doc in corpus if p in doc.split())
    return np.log(len(corpus) / df) if df > 0 else 0
# Calcular matriz TF-IDF
matriz = []
for doc in corpus:
    linha = [tf(p, doc) * idf(p, corpus) for p in vocab]
    matriz.append(linha)
df_tfidf = pd.DataFrame(matriz, columns=vocab)
print(df_tfidf.round(3))

   atendimento    bom  entrega  produto  quebrado  rapida   ruim
0        0.000  0.072    0.173    0.072     0.000   0.173  0.000
1        0.000  0.000    0.000    0.096     0.462   0.000  0.231
2        0.173  0.072    0.173    0.000     0.000   0.173  0.000
3        0.173  0.072    0.000    0.072     0.000   0.000  0.173


## Slide 52 — SAÍDA DO TF-IDF MANUAL

*Bloco 14*

In [ ]:
# Saída ilustrativa (arredondada):
#                atendimento    bom  entrega  produto  quebrado  rapida  ruim
# Doc 1               0.000     0.072  0.072    0.072     0.000   0.072  0.000
# Doc 2               0.000     0.000  0.000    0.096     0.462   0.000  0.231
# Doc 3               0.173     0.072  0.072    0.000     0.000   0.072  0.000
# Doc 4               0.173     0.072  0.000    0.072     0.000   0.000  0.231
# Repare como cada palavra recebe peso DIFERENTE em cada doc:
# - 'produto' aparece em 3/4 docs → IDF baixo → peso BAIXO
# - 'quebrado' aparece em 1/4 docs → IDF ALTO → peso 0.462 (TOP!)
# - 'atendimento' aparece em 2/4 docs → IDF médio
# A palavra MAIS DISCRIMINATIVA do Doc 2 é 'quebrado' — exatamente o que faz
# sentido! É a palavra que SINALIZA o problema do produto.

## Slide 53 — TFIDFVECTORIZER EM PRODUÇÃO

*Bloco 15*

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),         # unigrams + bigrams
    min_df=5,                   # ignora termos em <5 docs
    max_df=0.85,                # ignora termos em >85% dos docs
    max_features=10000,         # cap no vocabulário
    sublinear_tf=True,          # aplica log(1+TF) em vez de TF puro
)
X = vectorizer.fit_transform(reviews_corpus)  # 10k documentos
print(X.shape)        # (10000, 10000)
print(X.nnz)          # número de elementos não-zero
# Top features por documento
import numpy as np
feature_names = vectorizer.get_feature_names_out()
doc0 = X[0].toarray().flatten()
top_idx = np.argsort(doc0)[-5:][::-1]
print('Top 5 features do doc 0:')
for i in top_idx:
    print(f'  {feature_names[i]}: {doc0[i]:.3f}')

(400, 128)
5521
Top 5 features do doc 0:
  simples péssimo: 0.453
  horrível embalagem: 0.418
  péssimo produto: 0.276
  péssimo: 0.276
  atendimento horrível: 0.250


## Slide 55 — BOW VS TF-IDF: TAREFA REAL

*Bloco 16*

In [22]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
# Hipotético: reviews_corpus e labels (0=neg, 1=pos) já preparados
# 1) BoW
bow = CountVectorizer(ngram_range=(1, 2), min_df=5)
X_bow = bow.fit_transform(reviews_corpus)
score_bow = cross_val_score(
    LogisticRegression(max_iter=1000), X_bow, labels,
    cv=5, scoring='f1'
).mean()
# 2) TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(reviews_corpus)
score_tfidf = cross_val_score(
    LogisticRegression(max_iter=1000), X_tfidf, labels,
    cv=5, scoring='f1'
).mean()
print(f'BoW    F1: {score_bow:.3f}')   # ex: 0.812
print(f'TF-IDF F1: {score_tfidf:.3f}') # ex: 0.834 — geralmente vence

BoW    F1: 1.000
TF-IDF F1: 1.000
